In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_excel("campaign_experiment_data.xlsx")

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# ------------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------------
print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)

missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage (%)": (df.isnull().mean() * 100).round(2)
})

print(missing_summary[missing_summary["Missing Count"] > 0])

# ------------------------------------------------------------------
# Group Counts
# ------------------------------------------------------------------
print("\n" + "="*60)
print("EXPERIMENT GROUP COUNTS")
print("="*60)

print(df['experiment_group'].value_counts())

# ------------------------------------------------------------------
# Duplicate User IDs
# ------------------------------------------------------------------
print("\n" + "="*60)
print("DUPLICATE USER IDs")
print("="*60)

duplicate_count = df['user_id'].duplicated().sum()
print("Duplicate User IDs:", duplicate_count)

# ------------------------------------------------------------------
# Binary Value Validation
# ------------------------------------------------------------------
print("\n" + "="*60)
print("BINARY VALUE VALIDATION")
print("="*60)

binary_columns = [
    'visited_landing_page',
    'started_trial',
    'completed_onboarding',
    'converted_to_paid',
    'refund_requested'
]

for col in binary_columns:
    invalid_values = set(df[col].dropna().unique()) - {0,1}
    if len(invalid_values) == 0:
        print(f"{col}: Valid")
    else:
        print(f"{col}: Invalid values found -> {invalid_values}")

# ------------------------------------------------------------------
# Revenue Outliers using IQR
# ------------------------------------------------------------------
print("\n" + "="*60)
print("REVENUE OUTLIERS")
print("="*60)

Q1 = df['revenue_30d'].quantile(0.25)
Q3 = df['revenue_30d'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['revenue_30d'] < lower_bound) |
    (df['revenue_30d'] > upper_bound)
]

print("Number of Revenue Outliers:", len(outliers))

# ------------------------------------------------------------------
# Segment Distribution Across Groups
# ------------------------------------------------------------------
print("\n" + "="*60)
print("REGION DISTRIBUTION")
print("="*60)
print(pd.crosstab(df['region'], df['experiment_group']))

print("\n" + "="*60)
print("DEVICE TYPE DISTRIBUTION")
print("="*60)
print(pd.crosstab(df['device_type'], df['experiment_group']))

print("\n" + "="*60)
print("TRAFFIC SOURCE DISTRIBUTION")
print("="*60)
print(pd.crosstab(df['traffic_source'], df['experiment_group']))

print("\n" + "="*60)
print("PLAN TYPE DISTRIBUTION")
print("="*60)
print(pd.crosstab(df['plan_type'], df['experiment_group']))

DATASET OVERVIEW
Shape: (1408, 16)

Columns:
['user_id', 'signup_date', 'experiment_group', 'region', 'device_type', 'traffic_source', 'plan_type', 'visited_landing_page', 'started_trial', 'completed_onboarding', 'converted_to_paid', 'revenue_30d', 'support_tickets_30d', 'refund_requested', 'days_to_convert', 'engagement_score']

MISSING VALUES
                  Missing Count  Missing Percentage (%)
device_type                  18                    1.28
traffic_source               24                    1.70
days_to_convert            1336                   94.89
engagement_score             14                    0.99

EXPERIMENT GROUP COUNTS
experiment_group
Treatment    715
Control      693
Name: count, dtype: int64

DUPLICATE USER IDs
Duplicate User IDs: 8

BINARY VALUE VALIDATION
visited_landing_page: Valid
started_trial: Valid
completed_onboarding: Valid
converted_to_paid: Valid
refund_requested: Valid

REVENUE OUTLIERS
Number of Revenue Outliers: 72

REGION DISTRIBUTION
experime

In [2]:
# Create a copy for cleaning
cleaned_df = df.copy()

# Fill missing values
cleaned_df['device_type'] = cleaned_df['device_type'].fillna('Unknown')
cleaned_df['traffic_source'] = cleaned_df['traffic_source'].fillna('Unknown')

# Fill engagement_score with median
median_engagement = cleaned_df['engagement_score'].median()
cleaned_df['engagement_score'] = cleaned_df['engagement_score'].fillna(median_engagement)

# Keep days_to_convert missing values unchanged
# (Missing values correspond to users who never converted)

# Remove duplicate users
cleaned_df = cleaned_df.drop_duplicates(subset='user_id', keep='first')

# Create Data Quality Report
quality_report = pd.DataFrame({
    "Check": [
        "Missing device_type",
        "Missing traffic_source",
        "Missing engagement_score",
        "Missing days_to_convert",
        "Duplicate user IDs",
        "Invalid binary values",
        "Revenue outliers"
    ],
    "Result": [
        18,
        24,
        14,
        1336,
        8,
        0,
        72
    ],
    "Action Taken": [
        "Filled with Unknown",
        "Filled with Unknown",
        "Filled with Median",
        "Left unchanged",
        "Removed duplicates",
        "No action needed",
        "Retained"
    ]
})

# Save Excel workbook
with pd.ExcelWriter("experiment_analysis.xlsx") as writer:
    df.to_excel(writer, sheet_name="raw_data", index=False)
    cleaned_df.to_excel(writer, sheet_name="cleaned_data", index=False)
    quality_report.to_excel(writer, sheet_name="data_quality_checks", index=False)

print("experiment_analysis.xlsx created successfully.")
print("Cleaned dataset shape:", cleaned_df.shape)

experiment_analysis.xlsx created successfully.
Cleaned dataset shape: (1400, 16)
